In [18]:
import pandas as pd
from geopy.distance import geodesic
import numpy as np

# --- JOUW DATA ---
data = {
    'STN': [215, 235, 240, 249, 251, 260, 267],
    'LON(east)': [4.437, 4.781, 4.790, 4.979, 5.346, 5.180, 5.384],
    'LAT(north)': [52.141, 52.928, 52.318, 52.644, 53.392, 52.100, 52.898],
    'NAME': ['Voorschoten', 'De Kooy', 'Schiphol', 'Berkhout', 'Hoorn Terschelling', 'De Bilt', 'Stavoren'],
    'Temperature_C': [15.2, 14.8, 15.5, 14.9, 14.1, 15.6, 14.5],
    'Wind_Speed_kmh': [22, 28, 25, 26, 35, 19, 30]
}
df_stations = pd.DataFrame(data)

def compute_distances(input_lat, input_lon, stations_df):
    """Voegt kolom 'distance_km' toe met geodesische afstand."""
    input_location = (input_lat, input_lon)
    distances = stations_df.apply(
        lambda row: geodesic(input_location, (row['LAT(north)'], row['LON(east)'])).km,
        axis=1
    )
    out = stations_df.copy()
    out['distance_km'] = distances
    return out

def pick_k_nearest(stations_with_dist, k=3):
    """Selecteert k dichtstbijzijnde stations."""
    return stations_with_dist.sort_values('distance_km', ascending=True).head(k)

def idw_weights(distances_km, p=2.0, eps=1e-6):
    """
    IDW-gewichten: w_i = 1 / (d_i + eps)^p, daarna normaliseren.
    p = 1..3 (hoger = dichterbij weegt veel zwaarder).
    """
    raw = 1.0 / np.power(np.array(distances_km) + eps, p)
    w = raw / raw.sum()
    return w

def interpolate_at_point(input_lat, input_lon, stations_df,
                         value_cols=('Temperature_C', 'Wind_Speed_kmh'),
                         k=3, p=2.0, eps=1e-6, snap_threshold_m=10.0):
    """
    Interpoleert waarden op een punt met IDW over k dichtstbijzijnde stations.
    - value_cols: kolommen die je wilt middelen
    - p: IDW-power
    - eps: numerieke stabiliteit
    - snap_threshold_m: als d < threshold -> neem stationwaarde direct
    """
    # 1) Afstanden
    df_dist = compute_distances(input_lat, input_lon, stations_df)

    # 2) Edge case: exact op station?
    if df_dist['distance_km'].min() * 1000.0 < snap_threshold_m:
        row = df_dist.loc[df_dist['distance_km'].idxmin()]
        return {col: float(row[col]) for col in value_cols}, df_dist.sort_values('distance_km').head(k)

    # 3) k dichtstbij
    nearest = pick_k_nearest(df_dist, k=k)

    # 4) Gewichten
    w = idw_weights(nearest['distance_km'].to_numpy(), p=p, eps=eps)

    # 5) Gewogen gemiddelde per kolom
    result = {}
    for col in value_cols:
        vals = nearest[col].to_numpy(dtype=float)
        result[col] = float(np.dot(w, vals))

    return result, nearest

# --- VOORBEELD ---
my_lat = 51.9225   # Rotterdam Centraal
my_lon = 4.47917

interp, used_stations = interpolate_at_point(
    my_lat, my_lon, df_stations,
    value_cols=('Temperature_C', 'Wind_Speed_kmh'),
    k=3,        # aantal buren
    p=2.0,      # sterkte van de afstandsweging
    eps=1e-6,
    snap_threshold_m=10.0
)





print("Gebruikte stations (dichtstbij):")
print(used_stations[['NAME', 'STN', 'distance_km', 'Temperature_C', 'Wind_Speed_kmh']])
print("-" * 50)
print(f"Geschatte temperatuur: {interp['Temperature_C']:.2f} °C")
print(f"Geschatte windsnelheid: {interp['Wind_Speed_kmh']:.2f} km/h")






Gebruikte stations (dichtstbij):
          NAME  STN  distance_km  Temperature_C  Wind_Speed_kmh
0  Voorschoten  215    24.483698           15.2              22
2     Schiphol  240    48.886385           15.5              25
5      De Bilt  260    52.014654           15.6              19
--------------------------------------------------
Geschatte temperatuur: 15.31 °C
Geschatte windsnelheid: 22.06 km/h
